# Exp 009b Noisy Student Priors Postproc

## Goal

Check whether the proven `exp_007` metadata priors and texture-aware smoothing still add value on top of the first successful noisy-student branch (`exp_009`).

Success criteria:
- reconstruct the exact fold `0` validation split used by `exp_009`
- compare raw `exp_009` validation probabilities against the same postprocessing family used in `exp_005/exp_007/exp_008b`
- decide whether `exp_009` already absorbed most of the metadata signal or still benefits from the inference layer


## Expected Readout

The notebook saves:
- `ablation_results.csv`
- `classwise_auc_comparison.csv`
- `best_valid_predictions.parquet` or CSV fallback
- `result_snapshot.json`

Interpretation focus:
- does postprocessing still improve the first noisy-student fold?
- if yes, is the gain large enough to matter before moving to more folds?
- if no, does that suggest the student already internalized the soundscape prior signal?


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')


@dataclass
class EvalConfig:
    fold: int = 0
    n_folds: int = 5
    lambda_event: float = 0.4
    lambda_texture: float = 1.0
    smooth_texture: float = 0.35


CFG = EvalConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
EXP009_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_009_noisy_student_pseudolabel'
FOLD_DIR = EXP009_DIR / f'fold_{CFG.fold:02d}'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_009b_noisy_student_priors_postproc'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    'project_root': str(PROJECT_ROOT),
    'fold_dir': str(FOLD_DIR),
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
TEXTURE_TAXA = {'Amphibia', 'Insecta'}


def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(PRIMARY_LABELS), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)
sc_clean = sc_clean.sort_values(['filename', 'end_sec']).reset_index(drop=True)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].copy().reset_index(drop=True)

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))
_, valid_full_idx = full_splits[CFG.fold % len(full_splits)]
valid_files = set(full_df.iloc[valid_full_idx]['filename'].tolist())

train_df = (
    sc_clean[~sc_clean['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)
valid_df = (
    full_df[full_df['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)

texture_labels = taxonomy.loc[taxonomy['class_name'].isin(TEXTURE_TAXA), 'primary_label'].tolist()
texture_set = set(texture_labels)
idx_texture = np.array([label_to_idx[label] for label in PRIMARY_LABELS if label in texture_set], dtype=np.int32)
idx_event = np.array([idx for idx, label in enumerate(PRIMARY_LABELS) if label not in texture_set], dtype=np.int32)

{
    'all_labeled_segments': int(len(sc_clean)),
    'fully_labeled_files': int(len(full_files)),
    'valid_files_in_fold': int(len(valid_files)),
    'train_segments_for_priors': int(len(train_df)),
    'valid_segments': int(len(valid_df)),
    'texture_classes': int(len(idx_texture)),
    'event_classes': int(len(idx_event)),
}


In [ ]:
def stable_logit(p: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float32), eps, 1.0 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def stable_sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def score_macro_auc(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> dict:
    aucs = []
    skipped_no_positive = 0
    skipped_no_negative = 0
    for idx, _ in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0:
            skipped_no_positive += 1
            continue
        if negatives == 0:
            skipped_no_negative += 1
            continue
        aucs.append(float(roc_auc_score(y_true[:, idx], y_score[:, idx])))
    macro_auc = float(np.mean(aucs)) if aucs else float('nan')
    return {
        'macro_auc': macro_auc,
        'scored_classes': len(aucs),
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray) -> dict:
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return stable_logit(p, eps=eps)


def smooth_cols_by_file(scores: np.ndarray, meta_df: pd.DataFrame, cols: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    smoothed = scores.copy()
    ordered_meta = meta_df[['filename', 'end_sec']].reset_index(drop=True)
    for _, group_idx in ordered_meta.groupby('filename', sort=False).groups.items():
        group_idx = np.array(list(group_idx))
        if len(group_idx) <= 1:
            continue
        x = smoothed[group_idx][:, cols]
        prev_x = np.concatenate([x[:1], x[:-1]], axis=0)
        next_x = np.concatenate([x[1:], x[-1:]], axis=0)
        smoothed[np.ix_(group_idx, cols)] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return smoothed


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, meta_df: pd.DataFrame, lambda_event: float, lambda_texture: float, smooth_texture: float = 0.0) -> np.ndarray:
    fused = logits.copy()
    if len(idx_event):
        fused[:, idx_event] += lambda_event * prior_logits[:, idx_event]
    if len(idx_texture):
        fused[:, idx_texture] += lambda_texture * prior_logits[:, idx_texture]
    if smooth_texture > 0:
        fused = smooth_cols_by_file(fused, meta_df, idx_texture, alpha=smooth_texture)
    return fused


def classwise_auc_table(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> pd.DataFrame:
    rows = []
    for idx, label in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0 or negatives == 0:
            auc = np.nan
        else:
            auc = float(roc_auc_score(y_true[:, idx], y_score[:, idx]))
        rows.append({
            'primary_label': label,
            'positives': positives,
            'negatives': negatives,
            'auc': auc,
        })
    return pd.DataFrame(rows)


fold_snapshot = json.loads((FOLD_DIR / 'result_snapshot.json').read_text())
meta_df = pd.read_csv(FOLD_DIR / 'best_valid_meta.csv').sort_values(['filename', 'end_sec']).reset_index(drop=True)
outputs = np.load(FOLD_DIR / 'best_valid_outputs.npz')
y_true = outputs['y_true'].astype(np.float32)
raw_probs = outputs['y_pred'].astype(np.float32)
raw_logits = stable_logit(raw_probs)

if len(meta_df) != len(valid_df):
    raise ValueError(f'Exported meta rows do not match reconstructed validation fold: {len(meta_df)} vs {len(valid_df)}')
if meta_df['row_id'].tolist() != valid_df['row_id'].tolist():
    raise ValueError('row_id mismatch between exp_009 export and reconstructed validation fold')

prior_targets = np.stack([encode_multi_hot(labels) for labels in train_df['label_list']], axis=0)
prior_tables = fit_prior_tables(train_df[['site', 'hour_utc']], prior_targets)
prior_logits = prior_logits_from_tables(
    sites=meta_df['site'].to_numpy(),
    hours=meta_df['hour_utc'].to_numpy(),
    tables=prior_tables,
)

variant_settings = [
    ('raw', None),
    ('event_priors_only', (CFG.lambda_event, 0.0, 0.0)),
    ('texture_priors_only', (0.0, CFG.lambda_texture, 0.0)),
    ('event_texture_priors', (CFG.lambda_event, CFG.lambda_texture, 0.0)),
    ('event_texture_priors_smooth', (CFG.lambda_event, CFG.lambda_texture, CFG.smooth_texture)),
]

variant_scores = {}
rows = []
for name, settings in variant_settings:
    if settings is None:
        probs = raw_probs.copy()
    else:
        lambda_event, lambda_texture, smooth_texture = settings
        fused_logits = fuse_native_logits(
            logits=raw_logits,
            prior_logits=prior_logits,
            meta_df=meta_df,
            lambda_event=lambda_event,
            lambda_texture=lambda_texture,
            smooth_texture=smooth_texture,
        )
        probs = stable_sigmoid(fused_logits)
    variant_scores[name] = probs
    metrics = score_macro_auc(y_true, probs, PRIMARY_LABELS)
    rows.append({'variant': name, **metrics})

ablation_df = pd.DataFrame(rows).sort_values('macro_auc', ascending=False).reset_index(drop=True)
display(ablation_df)


In [ ]:
best_variant = ablation_df.iloc[0]['variant']
raw_macro_auc = float(ablation_df.loc[ablation_df['variant'] == 'raw', 'macro_auc'].iloc[0])
best_macro_auc = float(ablation_df.iloc[0]['macro_auc'])
best_probs = variant_scores[best_variant]

classwise_raw = classwise_auc_table(y_true, raw_probs, PRIMARY_LABELS).rename(columns={'auc': 'raw_auc'})
classwise_best = classwise_auc_table(y_true, best_probs, PRIMARY_LABELS).rename(columns={'auc': 'best_auc'})
classwise_compare = classwise_raw.merge(classwise_best[['primary_label', 'best_auc']], on='primary_label', how='left')
classwise_compare['delta'] = classwise_compare['best_auc'] - classwise_compare['raw_auc']
classwise_compare.to_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv', index=False)

ablation_df.to_csv(OUTPUT_DIR / 'ablation_results.csv', index=False)

prediction_export = pd.concat([
    meta_df.reset_index(drop=True),
    pd.DataFrame({'fold': [int(CFG.fold)] * len(meta_df), 'best_variant': [best_variant] * len(meta_df)}),
    pd.DataFrame(y_true.astype(np.int8), columns=[f'true__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(raw_probs.astype(np.float32), columns=[f'raw__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(best_probs.astype(np.float32), columns=[f'{best_variant}__{label}' for label in PRIMARY_LABELS]),
], axis=1)
try:
    prediction_export.to_parquet(OUTPUT_DIR / 'best_valid_predictions.parquet', index=False)
except Exception:
    prediction_export.to_csv(OUTPUT_DIR / 'best_valid_predictions.csv', index=False)

result_snapshot = {
    'experiment_id': 'exp_009b',
    'experiment_name': 'noisy_student_priors_postproc',
    'fold': int(CFG.fold),
    'source_experiment': 'exp_009',
    'source_best_epoch': int(fold_snapshot.get('best_epoch')) if fold_snapshot.get('best_epoch') is not None else None,
    'raw_macro_auc': raw_macro_auc,
    'best_variant': str(best_variant),
    'best_macro_auc': best_macro_auc,
    'delta_vs_raw': best_macro_auc - raw_macro_auc,
    'lambda_event': float(CFG.lambda_event),
    'lambda_texture': float(CFG.lambda_texture),
    'smooth_texture': float(CFG.smooth_texture),
    'scored_classes_best_variant': int(ablation_df.iloc[0]['scored_classes']),
    'valid_rows': int(len(meta_df)),
    'pseudo_rows': int(fold_snapshot.get('pseudo_rows', 0)),
    'pseudo_files': int(fold_snapshot.get('pseudo_files', 0)),
    'pseudo_confidence_mean': float(fold_snapshot.get('pseudo_confidence_mean', 0.0)),
}
(OUTPUT_DIR / 'result_snapshot.json').write_text(json.dumps(result_snapshot, indent=2))

print('Snapshot:')
print(json.dumps(result_snapshot, indent=2))


In [ ]:
snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
print('Snapshot:')
print(json.dumps(snapshot, indent=2))

print()
print('Ablations:')
display(pd.read_csv(OUTPUT_DIR / 'ablation_results.csv'))

compare_df = pd.read_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv')
print()
print('Top gains:')
display(compare_df.sort_values('delta', ascending=False).head(20))
print()
print('Top regressions:')
display(compare_df.sort_values('delta', ascending=True).head(20))


## Reading The Result

This notebook is intentionally narrow.

If priors help again here, the noisy-student branch is not only a stronger training recipe; it is also compatible with the same soundscape-aware inference layer that already transferred across earlier native branches.

If priors help only a little, that is still useful information: it means `exp_009` may already be internalizing more of the metadata and texture structure during training than the earlier supervised branches did.
